<a href="https://colab.research.google.com/github/anaclaracandidoo/projetodeep-learning/blob/main/04_api_adaptacao_material.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Instalação das bibliotecas de backend e IA
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install -q ollama langchain-community langchain-core fastapi uvicorn httpx nest_asyncio

# 2. Inicialização do servidor Ollama em segundo plano
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 3. Download do modelo LLM
!ollama pull qwen2.5:3b

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [73.2 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu noble/main all Packages [10.3 MB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [52.1 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,037 kB]
Get:

In [2]:
import nest_asyncio
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel

# Permite executar loop assíncrono dentro do Jupyter/Colab
nest_asyncio.apply()

# 1. Inicialização do App FastAPI e do Modelo
app = FastAPI(title="API de Adaptação de Material EduAcessível")
llm = Ollama(model="qwen2.5:3b", temperature=0.2)

# 2. Modelos de dados para Requisição e Resposta (Pydantic)
class MaterialRequest(BaseModel):
    texto_original: str
    tipo_adaptacao: str  # Ex: "Resumo Simplificado", "Linguagem Clara", "Tópicos Principais"

class MaterialResponse(BaseModel):
    texto_adaptado: str
    tipo_adaptacao: str

# 3. Definição do Endpoint da API
@app.post("/api/adaptar", response_model=MaterialResponse)
def adaptar_material(payload: MaterialRequest):
    try:
        template = """
        Você é um assistente pedagógico especializado em acessibilidade educacional.
        Sua tarefa é adaptar o texto abaixo de acordo com a solicitação: '{tipo_adaptacao}'.

        Texto Original:
        {texto_original}

        Texto Adaptado:
        """
        prompt = ChatPromptTemplate.from_template(template)
        chain = prompt | llm

        resultado = chain.invoke({
            "tipo_adaptacao": payload.tipo_adaptacao,
            "texto_original": payload.texto_original
        })

        return MaterialResponse(
            texto_adaptado=resultado.strip(),
            tipo_adaptacao=payload.tipo_adaptacao
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

# 4. Teste local da API utilizando TestClient do FastAPI
client = TestClient(app)

payload_teste = {
    "texto_original": "A fotossíntese é o processo físico-químico, a nível celular, realizado pelos seres vivos clorofilados, que utilizam a luz solar como fonte de energia para a síntese de matéria orgânica a partir de dióxido de carbono e água.",
    "tipo_adaptacao": "Linguagem Clara e Simplificada"
}

print("--- Enviando requisição para a API (/api/adaptar) ---")
response = client.post("/api/adaptar", json=payload_teste)

print(f"Status Code: {response.status_code}")
print("Resposta da API:")
print(response.json()["texto_adaptado"])

/tmp/ipykernel_3293/210786241.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama
/tmp/ipykernel_3293/210786241.py:13: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="qwen2.5:3b", temperature=0.2)


--- Enviando requisição para a API (/api/adaptar) ---
Status Code: 200
Resposta da API:
Texto Adaptado:
A fotossíntese é um processo que ocorre nas células de plantas e outros seres vivos que contêm clorofila. Este processo usa a luz solar como fonte de energia para criar substâncias orgânicas a partir de dióxido de carbono e água.
